In [2]:
# ============================================================
# CELL 1 - GPU CHECK
# ============================================================

import torch

print("PyTorch:", torch.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
else:
    print("GPU no disponible")

PyTorch: 2.11.0+cu130
GPU: Tesla T4
CUDA: 13.0


In [3]:
# ============================================================
# CELL 2 - Instalación de dependencias
# ============================================================

!pip -q install transformers
!pip -q install sentence-transformers
!pip -q install datasets
!pip -q install evaluate
!pip -q install accelerate
!pip -q install huggingface_hub
!pip -q install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


In [4]:
# ============================================================
# CELL 2.1 - Verificacion de instalacion
# ============================================================

import torch
import transformers
import datasets
import sentence_transformers
import sklearn

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("Scikit-Learn:", sklearn.__version__)

print("\nGPU Available:", torch.cuda.is_available())

Torch: 2.11.0+cu130
Transformers: 5.18.0
Datasets: 4.8.5
Sentence Transformers: 5.7.0
Scikit-Learn: 1.6.1

GPU Available: True


In [5]:
# ==========================================
# 3. AUTENTICACIÓN HUGGING FACE
# ==========================================

from huggingface_hub import notebook_login

notebook_login()

In [6]:
# ==========================================
# 5. CARGA DEL DATASET AG NEWS
# ==========================================

from datasets import load_dataset

dataset = load_dataset("fancyzhx/ag_news")

print(dataset)

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [7]:
# ==========================================
# VISUALIZAR EJEMPLOS DEL DATASET
# ==========================================

print(dataset["train"][0])

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


In [9]:
# ==========================================
# ETIQUETAS DEL DATASET
# ==========================================

labels = dataset["train"].features["label"].names

print(labels)

['World', 'Sports', 'Business', 'Sci/Tech']


In [8]:
# ==========================================
# TAMAÑO DEL DATASET
# ==========================================

print(dataset)

print("\nTrain:", len(dataset["train"]))
print("Test:", len(dataset["test"]))

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

Train: 120000
Test: 7600


In [10]:
# ============================================================
# MODELO 1 - FACEBOOK BART LARGE MNLI
# ============================================================
# Este modelo será utilizado para clasificar noticias mediante
# Zero-Shot Classification.
#
# Ventajas:
# - Alta precisión
# - Muy utilizado en NLP
# - Excelente comprensión semántica
#
# Desventaja:
# - Mayor consumo de memoria y mayor latencia

from transformers import pipeline

bart_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0
)

print("Modelo BART cargado correctamente.")

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Modelo BART cargado correctamente.


In [11]:
# ============================================================
# MODELO 2 - DEBERTA V3 ZERO SHOT
# ============================================================
# Modelo optimizado para tareas de clasificación.
#
# Ventajas:
# - Buen equilibrio entre precisión y velocidad
# - Menor consumo de recursos
#
# Desventaja:
# - Puede ser ligeramente menos preciso que modelos más grandes

from transformers import pipeline

deberta_classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/deberta-v3-base-zeroshot-v1.1-all-33",
    device=0
)

print("Modelo DeBERTa cargado correctamente.")

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  369MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

...

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Modelo DeBERTa cargado correctamente.


In [12]:
# ============================================================
# MODELO 3 - MODERNBERT NLI
# ============================================================
# Modelo moderno entrenado para tareas de inferencia
# de lenguaje natural.
#
# Ventajas:
# - Excelente comprensión de contexto
# - Buen desempeño en clasificación textual
#
# Desventaja:
# - Latencia superior a modelos ligeros

from transformers import pipeline

modernbert_classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/ModernBERT-large-zeroshot-v2.0",
    device=0
)

print("Modelo ModernBERT cargado correctamente.")

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  792MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/174 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.58M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Modelo ModernBERT cargado correctamente.


In [14]:
# ============================================================
# CREAR MUESTRA DE EVALUACIÓN
# ============================================================

sample_test = dataset["test"].shuffle(seed=42).select(range(500))

print("Registros seleccionados:", len(sample_test))


Registros seleccionados: 500


In [15]:
# ============================================================
# PIPELINE DE INFERENCIA - MODELO 1
# Facebook BART Large MNLI
# ============================================================

candidate_labels = [
    "World",
    "Sports",
    "Business",
    "Sci/Tech"
]

texto_prueba = sample_test[0]["text"]

resultado_bart = bart_classifier(
    texto_prueba,
    candidate_labels=candidate_labels
)

print(resultado_bart)

{'sequence': 'Indian board plans own telecast of Australia series The Indian cricket board said on Wednesday it was making arrangements on its own to broadcast next month #39;s test series against Australia, which is under threat because of a raging TV rights dispute.', 'labels': ['Sports', 'Sci/Tech', 'Business', 'World'], 'scores': [0.44296377897262573, 0.24426588416099548, 0.16293354332447052, 0.14983682334423065]}


In [16]:
# ============================================================
# PIPELINE DE INFERENCIA - MODELO 2
# DeBERTa V3 Zero-Shot
# ============================================================

candidate_labels = [
    "World",
    "Sports",
    "Business",
    "Sci/Tech"
]

texto_prueba = sample_test[0]["text"]

resultado_deberta = deberta_classifier(
    texto_prueba,
    candidate_labels=candidate_labels
)

print(resultado_deberta)

{'sequence': 'Indian board plans own telecast of Australia series The Indian cricket board said on Wednesday it was making arrangements on its own to broadcast next month #39;s test series against Australia, which is under threat because of a raging TV rights dispute.', 'labels': ['Sports', 'World', 'Sci/Tech', 'Business'], 'scores': [0.9884923696517944, 0.011255299672484398, 0.0001524127583252266, 9.995473374146968e-05]}


In [17]:
# ============================================================
# PIPELINE DE INFERENCIA - MODELO 3
# ModernBERT Large Zero-Shot
# ============================================================

candidate_labels = [
    "World",
    "Sports",
    "Business",
    "Sci/Tech"
]

texto_prueba = sample_test[0]["text"]

resultado_modernbert = modernbert_classifier(
    texto_prueba,
    candidate_labels=candidate_labels
)

print(resultado_modernbert)

{'sequence': 'Indian board plans own telecast of Australia series The Indian cricket board said on Wednesday it was making arrangements on its own to broadcast next month #39;s test series against Australia, which is under threat because of a raging TV rights dispute.', 'labels': ['Sports', 'World', 'Business', 'Sci/Tech'], 'scores': [0.9452512860298157, 0.045258406549692154, 0.008705463260412216, 0.0007848214590921998]}


# Cálculo de métricas (accuracy, precision, recall, F1-score).

In [19]:
# ============================================================
# MUESTRA DE EVALUACIÓN
# ============================================================

sample_test = dataset["test"].shuffle(seed=42).select(range(300))

print("Registros seleccionados:", len(sample_test))

Registros seleccionados: 300


In [21]:
# ============================================================
# FUNCIÓN DE EVALUACIÓN
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

label_map = {
    "World": 0,
    "Sports": 1,
    "Business": 2,
    "Sci/Tech": 3
}


def evaluate_model(classifier, dataset_sample):

    y_true = []
    y_pred = []

    for row in dataset_sample:

        prediction = classifier(
            row["text"],
            candidate_labels=list(label_map.keys())
        )

        predicted_label = prediction["labels"][0]

        y_true.append(row["label"])
        y_pred.append(label_map[predicted_label])

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    return accuracy, precision, recall, f1

In [22]:
# ============================================================
# EVALUACIÓN - BART LARGE MNLI
# ============================================================

bart_accuracy, bart_precision, bart_recall, bart_f1 = evaluate_model(
    bart_classifier,
    sample_test
)

print("BART")
print("Accuracy :", round(bart_accuracy,4))
print("Precision:", round(bart_precision,4))
print("Recall   :", round(bart_recall,4))
print("F1 Score :", round(bart_f1,4))

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


BART
Accuracy : 0.6767
Precision: 0.684
Recall   : 0.6767
F1 Score : 0.6763


In [23]:
# ============================================================
# EVALUACIÓN - DEBERTA V3
# ============================================================

deberta_accuracy, deberta_precision, deberta_recall, deberta_f1 = evaluate_model(
    deberta_classifier,
    sample_test
)

print("DeBERTa")
print("Accuracy :", round(deberta_accuracy,4))
print("Precision:", round(deberta_precision,4))
print("Recall   :", round(deberta_recall,4))
print("F1 Score :", round(deberta_f1,4))

DeBERTa
Accuracy : 0.84
Precision: 0.8437
Recall   : 0.84
F1 Score : 0.8365


In [24]:
# ============================================================
# EVALUACIÓN - MODERNBERT
# ============================================================

modern_accuracy, modern_precision, modern_recall, modern_f1 = evaluate_model(
    modernbert_classifier,
    sample_test
)

print("ModernBERT")
print("Accuracy :", round(modern_accuracy,4))
print("Precision:", round(modern_precision,4))
print("Recall   :", round(modern_recall,4))
print("F1 Score :", round(modern_f1,4))

ModernBERT
Accuracy : 0.8067
Precision: 0.8223
Recall   : 0.8067
F1 Score : 0.7988


In [25]:
# ============================================================
# COMPARACIÓN DE RESULTADOS
# ============================================================

import pandas as pd

results = pd.DataFrame({
    "Modelo": [
        "BART Large MNLI",
        "DeBERTa V3",
        "ModernBERT"
    ],
    "Accuracy": [
        bart_accuracy,
        deberta_accuracy,
        modern_accuracy
    ],
    "Precision": [
        bart_precision,
        deberta_precision,
        modern_precision
    ],
    "Recall": [
        bart_recall,
        deberta_recall,
        modern_recall
    ],
    "F1 Score": [
        bart_f1,
        deberta_f1,
        modern_f1
    ]
})

results

,Modelo,Accuracy,Precision,Recall,F1 Score
0,BART Large MNLI,0.676667,0.684030,0.676667,0.676277
1,DeBERTa V3,0.840000,0.843719,0.840000,0.836494
2,ModernBERT,0.806667,0.822339,0.806667,0.798790


In [27]:
# ============================================================
# FUNCIÓN PARA MEDIR LATENCIA
# ============================================================

import time

def measure_latency(classifier, dataset_sample):

    start_time = time.time()

    for row in dataset_sample:
        classifier(
            row["text"],
            candidate_labels=[
                "World",
                "Sports",
                "Business",
                "Sci/Tech"
            ]
        )

    end_time = time.time()

    total_time = end_time - start_time
    avg_latency = total_time / len(dataset_sample)

    return total_time, avg_latency

In [28]:
# ============================================================
# LATENCIA - BART
# ============================================================

bart_total_time, bart_avg_latency = measure_latency(
    bart_classifier,
    sample_test
)

print("BART")
print("Tiempo total:", round(bart_total_time, 2), "segundos")
print("Latencia promedio:", round(bart_avg_latency, 4), "segundos")

BART
Tiempo total: 32.24 segundos
Latencia promedio: 0.1075 segundos


In [29]:
# ============================================================
# LATENCIA - DEBERTA
# ============================================================

deberta_total_time, deberta_avg_latency = measure_latency(
    deberta_classifier,
    sample_test
)

print("DeBERTa")
print("Tiempo total:", round(deberta_total_time, 2), "segundos")
print("Latencia promedio:", round(deberta_avg_latency, 4), "segundos")

DeBERTa
Tiempo total: 43.15 segundos
Latencia promedio: 0.1438 segundos


In [30]:
# ============================================================
# LATENCIA - MODERNBERT
# ============================================================

modern_total_time, modern_avg_latency = measure_latency(
    modernbert_classifier,
    sample_test
)

print("ModernBERT")
print("Tiempo total:", round(modern_total_time, 2), "segundos")
print("Latencia promedio:", round(modern_avg_latency, 4), "segundos")

ModernBERT
Tiempo total: 47.57 segundos
Latencia promedio: 0.1586 segundos


In [31]:
# ============================================================
# TABLA COMPARATIVA DE LATENCIA
# ============================================================

latency_results = pd.DataFrame({
    "Modelo": [
        "BART Large MNLI",
        "DeBERTa V3",
        "ModernBERT"
    ],
    "Tiempo Total (s)": [
        bart_total_time,
        deberta_total_time,
        modern_total_time
    ],
    "Latencia Promedio (s)": [
        bart_avg_latency,
        deberta_avg_latency,
        modern_avg_latency
    ]
})

latency_results

,Modelo,Tiempo Total (s),Latencia Promedio (s)
0,BART Large MNLI,32.235302,0.107451
1,DeBERTa V3,43.151147,0.143837
2,ModernBERT,47.570681,0.158569
